# 第 7 章 · 多 agent（agent-as-tool）

把**一个 agent 包装成工具**，给另一个 agent 调用。

和 Coordinator 的区别：那里是"把对话整个转交"，这里是"像调函数一样调，拿回值继续干自己的活"。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break

# 通用脚手架都在 common 里：run_once / run_session / run_async / 重试 / 静音
from common import (adk_model, quiet_adk_sync_warnings, retry_call,
                    run_once, run_session, run_async)
quiet_adk_sync_warnings()

try:
    import google.adk  # noqa: F401  —— 只为内核检查
except ModuleNotFoundError:
    raise RuntimeError(
        "本 notebook 需要内核『Python (agentic-adk)』，当前内核里没有 google.adk。\n"
        "VS Code 里点右上角内核名 → 选 Python (agentic-adk) → 重新运行本格。"
    ) from None

from google.adk.agents import LlmAgent
from google.adk.tools import agent_tool


def generate_image(prompt: str) -> dict:
    """Generates an image based on a textual prompt.

    Args:
        prompt: A detailed description of the image to generate.

    Returns:
        A dictionary with the status and a short description of the image.
    """
    # 【适配】书里返回 mock 的 image bytes；这里返回文字描述，方便在 notebook 里看结果
    print(f"TOOL: Generating image for prompt: '{prompt}'")
    return {"status": "success", "description": f"已生成图片：{prompt}"}


image_generator_agent = LlmAgent(
    name="ImageGen",
    model=adk_model(),                      # 【适配】gemini-2.0-flash 已下线
    description="Generates an image based on a detailed text prompt.",
    instruction="你是图像生成专家。把用户的整个请求当作 prompt 传给 generate_image 工具，"
                "然后把工具返回的描述原样告诉我。",
    tools=[generate_image],
)

# ★ 本章新增 AgentTool：把一个 agent 变成另一个 agent 可调用的工具
# 【适配】ADK 2.13 的 AgentTool 不收 description=，描述改由子 agent 自己的 description 提供
image_tool = agent_tool.AgentTool(agent=image_generator_agent)

artist_agent = LlmAgent(
    name="Artist",
    model=adk_model(),                      # 【适配】同上
    instruction="你是创意艺术家。先想一个有创意的画面描述，然后用 ImageGen 工具生成它。",
    tools=[image_tool],
)

finals, state = run_once(artist_agent, "画一只戴礼帽的猫")
for author, text in finals:
    print(f"\n--- [{author}] ---\n{text[:400]}")

TOOL: Generating image for prompt: 'A portrait of an elegant British Shorthair cat wearing a sophisticated black velvet top hat, sitting on an antique mahogany desk in a dimly lit, cozy Victorian-style library. Warm, soft golden light from a nearby desk lamp illuminates the cat's whiskers and the texture of the hat. In the background, blurred bookshelves filled with old leather-bound books. Highly detailed, cinematic lighting, artistic photography style.'



--- [Artist] ---
我为你创作了一幅“绅士猫”的肖像。

画面中，一只优雅的英国短毛猫头戴精致的黑色丝绒高礼帽，端坐在充满维多利亚时代复古气息的图书馆书桌上。柔和的暖光洒在它身上，背景是带着些许神秘感、模糊的书架，整体氛围像极了一位正要解开谜题的猫咪侦探。希望你喜欢这个充满故事感的画面！


## 补充：两个常见问题

1. **被拉起的子 agent 要处理一段特定数据（比如 state 里的值），怎么给？**
2. **子 agent 作为工具产出的字符串，主 agent 怎么消费？**

下面两格分别演示。

In [2]:
# ===== 问题 1：子 agent 怎么拿到"特定数据"（session.state）=====
from google.adk.tools import ToolContext


def read_state(tool_context: ToolContext) -> str:
    """读取 session.state 里的 data 字段。"""
    return f"STATE_DATA={tool_context.state.get('data', '(空)')}"


def probe_state_access(label, child_instruction, child_tools, seed):
    """建一个父 agent，把子 agent 包成工具，看子 agent 拿到了什么。"""
    child = LlmAgent(name="Child", model=adk_model(),
                     description="处理数据，回一行 RESULT=<值>。",
                     instruction=child_instruction, tools=child_tools)
    parent = LlmAgent(name="Parent", model=adk_model(),
                      instruction="调用 Child 工具，把它的返回值原样放进最终回答。",
                      tools=[agent_tool.AgentTool(agent=child)])

    runner = InMemoryRunner(parent)
    sid = str(uuid.uuid4())
    # ★ 关键：state 在建 session 时预置；子 agent 与父共用同一个 session
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid, state=seed)

    print(f"\n[{label}]  预置 state = {seed}")
    for ev in retry_call(lambda: list(runner.run(
            user_id="u", session_id=sid,
            new_message=types.Content(role="user", parts=[types.Part(text="处理数据")]))), label="run "):
        if not ev.content:
            continue
        for p in (ev.content.parts or []):
            if getattr(p, "function_response", None):
                print(f"   子 agent 返回给父 agent 的 result = {str(p.function_response.response)[:120]}")


# 方式 A：instruction 里用 {data} 占位符 —— 直接注入
probe_state_access("A 占位符 {data}", "只回一行：RESULT={data}", [], {"data": "机密42"})

# 方式 B：给子 agent 一个能读 state 的工具
probe_state_access("B 读 state 的工具",
                   "先用 read_state 工具取值，再回一行：RESULT=<值>",
                   [read_state], {"data": "机密42"})

# 两种都能拿到数据。机制差别：A 是【直接注入】（占位符被替换成 state 原值），
#   B 是【经模型转述】（工具返回值 → 模型再写一遍）。B 有失真的风险：
#   相同实验里曾出现模型把 '机密42' 写成 '42'（丢了前缀），本次没有。
#   要精确传递数据，优先用 A。


[A 占位符 {data}]  预置 state = {'data': '机密42'}


   子 agent 返回给父 agent 的 result = {'result': 'RESULT=机密42'}

[B 读 state 的工具]  预置 state = {'data': '机密42'}


   子 agent 返回给父 agent 的 result = {'result': 'RESULT=机密42'}


In [3]:
# ===== 问题 2：子 agent 返回的字符串怎么被主 agent 消费 =====
# 子 agent 只回一行"机器可读"的结果，父 agent 拿去用。


def run_and_show(agent, message, seed=None):
    """跑一次，把【工具返回】和【父 agent 的最终答复】分开打印。"""
    runner = InMemoryRunner(agent)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid, state=seed or {})
    final = ""
    for ev in retry_call(lambda: list(runner.run(
            user_id="u", session_id=sid,
            new_message=types.Content(role="user", parts=[types.Part(text=message)]))), label="run "):
        if not ev.content:
            continue
        for p in (ev.content.parts or []):
            if getattr(p, "function_response", None):
                print(f"  [工具返回] {p.function_response.name} -> {str(p.function_response.response)[:160]}")
            if getattr(p, "text", None) and p.text.strip():
                final = p.text.strip()
    print(f"  [主 agent 最终答复] {final[:300]}")


sentiment_agent = LlmAgent(
    name="SentimentProbe",
    model=adk_model(),
    description="判断一段文本的情绪，返回 SENTIMENT=positive|negative|neutral。",
    instruction="判断用户给的文本的情绪，只回一行：SENTIMENT=<positive|negative|neutral>",
)

# ★ 父 agent 的 instruction 里写清"怎么用返回值" —— 这是消费的关键
reviewer_agent = LlmAgent(
    name="Reviewer",
    model=adk_model(),
    instruction=(
        "先调用 SentimentProbe 工具判断情绪，然后写一句中文回复建议。"
        "你的回复里必须原样带上它返回的 SENTIMENT 值（例如 'SENTIMENT=negative'）。"
    ),
    tools=[agent_tool.AgentTool(agent=sentiment_agent)],
)

run_and_show(reviewer_agent, "这次交付又延期了，我很失望。")

  [工具返回] SentimentProbe -> {'result': 'SENTIMENT=negative'}
  [主 agent 最终答复] SENTIMENT=negative。对于这次延期给您带来的不便和失望，我们深表歉意；建议您立即与项目负责人对接，明确最新的交付时间及补偿方案，以减少后续损失。
